# Train YOLO26 segmentation on CribHD + Hackathon negatives

Derived from `train_cribhd_yolo26.ipynb`. Same toy classes, training settings and evaluation pipeline, with the requested Hackathon images added as **training-only negatives**. No source annotations are required: empty YOLO label files are generated only in the prepared copy. This treats these images as containing no toys, as requested; a negative must not contain an unannotated toy.

Original datasets, notebook and existing checkpoints are preserved. All outputs use `training_runs_hackathon_negatives` and UUID run names. Validation/test splits are not supplemented with these training images; rejection of unseen clothing still needs independent validation examples.

Adapted from [README_cribnet.md](README_cribnet.md) and `TOY-FILTER-Pipeline.ipynb`.
Select the Python kernel at the repository’s Python environment with the required libraries installed.
Install Ultralytics in that environment before running. No Detectron2, MiDaS or Colab is required.

The original toy notebook configures Mask R-CNN but loads a previously trained checkpoint; it does not call a training loop. Here we explicitly train, validate, and test YOLO26-seg. This follows the README's alternative YOLO workflow, not an exact reproduction of its YOLOv8/Detectron2 experiments.

**Default:** CribHD-T, with two classes: `hard-toy` and `soft-toy` (not the original notebook's hardcoded three classes). The local YOLO labels are polygons. `yolo26n.pt` is detection-only; use `yolo26n-seg.pt` for masks.

Training is not run when this notebook is created. Run the cells yourself. The model-loading cell downloads pretrained weights if absent. Dataset and inference remain local.

References: [CribNet](https://github.com/ostadabbas/CribNet), [YOLO segmentation](https://docs.ultralytics.com/tasks/segment/). Retain the original README's attribution and non-commercial-use notice when using its material.


In [ ]:
# Resolve paths from the cloned repository (run the notebook from its root).
from repo_paths import find_repo_root
REPO_ROOT = find_repo_root()
from pathlib import Path
from collections import Counter
import hashlib
import json
import math
import random
import time
import uuid

import cv2
import numpy as np
import torch
import yaml
from PIL import Image, ImageOps
from IPython.display import display
from ultralytics import YOLO

PROJECT = REPO_ROOT
DATASET = (REPO_ROOT / "CribHD/CribHD/CribHD-T/YoloV8")
# To train a separate blanket model, change CribHD-T above to CribHD-B.
RUN_NAME = f"cribhd_t_hackathon_negatives_{time.strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex}"
RUNS = PROJECT / "training_runs_hackathon_negatives"
NEGATIVE_DATA = (REPO_ROOT / "Dataset_Hackathon-20261009T184820Z-1-001")
PRETRAINED = PROJECT / "test2_models/yolo26n-seg.pt"
EPOCHS = 100                 # Maximum budget; early stopping still applies
IMAGE_SIZE = 640           # Compare 960/1280 later if small-object recall is poor
BATCH = 4                  # Reduce to 2 or 1 if GPU memory is insufficient
DEVICE = 0 if torch.cuda.is_available() else "cpu"
SEED = 42
# Optional reviewed examples: train/ and valid/, each with images/ and labels/.
EXTRA_DATA = (REPO_ROOT / "toy_reviewed_examples")
TARGET_PRECISION = 0.90
MIN_RECALL = 0.60  # Explicit tradeoff, not a guarantee for unseen camera scenes
print("Device:", torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU (training may be slow)")
print("Dataset:", DATASET)

## Add reviewed clothing examples before retraining
Put representative images in `toy_reviewed_examples/train/images` and separate validation scenes in `valid/images`. Each image must have a same-stem `.txt` file in the matching `labels` folder.

- Truly toy-free clothing/bedding scenes: create an **empty label file** after checking the entire image.
- Clothing with toys: annotate **every toy** with YOLO segmentation polygons, class 0 hard-toy or class 1 soft-toy. Leave clothing unannotated.
- Keep frames from the same recording/session in one split; do not move validation failures into training and continue scoring on them. Do not reuse test scenes.
- Missing labels are rejected rather than silently declaring an image toy-free. Never automatically relabel CribHD-B as background: its images can contain toys.

The following cell builds a separate combined copy only when extra images exist. Original CribHD files are preserved. The audit below checks the combined data. More epochs alone cannot guarantee fewer clothing detections.

In [ ]:
import shutil
import uuid

SOURCE_DATASET = (REPO_ROOT / "CribHD/CribHD/CribHD-T/YoloV8")
DATASET = SOURCE_DATASET
extra_pairs = []
for split in ("train", "valid"):
    image_dir = EXTRA_DATA / split / "images"
    label_dir = EXTRA_DATA / split / "labels"
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)
    for path in sorted(image_dir.iterdir()):
        if path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
            continue
        label = label_dir / (path.stem + ".txt")
        assert label.is_file(), f"Review and label this image first: {path}"
        extra_pairs.append((split, path, label))
if extra_pairs:
    DATASET = RUNS / "prepared_data" / f"{RUN_NAME}_{uuid.uuid4().hex[:8]}"
    shutil.copytree(SOURCE_DATASET, DATASET, ignore=shutil.ignore_patterns("*.cache"))
    for i, (split, path, label) in enumerate(extra_pairs):
        stem = f"reviewed_{i}_{uuid.uuid4().hex[:8]}"
        shutil.copy2(path, DATASET / split / "images" / (stem + path.suffix.lower()))
        shutil.copy2(label, DATASET / split / "labels" / (stem + ".txt"))
    print(f"Added {len(extra_pairs)} reviewed images. Combined data: {DATASET}")
else:
    print("No optional reviewed examples supplied; adding Hackathon negatives next.")

# Add the requested unlabeled folder as training-only background examples.
assert NEGATIVE_DATA.is_dir(), f"Missing negative-image folder: {NEGATIVE_DATA}"
negative_paths = sorted(p for p in NEGATIVE_DATA.rglob("*")
                        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
assert negative_paths, "No supported negative images found"
if DATASET == SOURCE_DATASET:
    DATASET = RUNS / "prepared_data" / f"{RUN_NAME}_{uuid.uuid4().hex}"
    shutil.copytree(SOURCE_DATASET, DATASET, ignore=shutil.ignore_patterns("*.cache"))

# Pixel hashing also catches identical decoded images with different metadata.
def image_digest(path):
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        return hashlib.sha256(str(im.size).encode() + im.tobytes()).hexdigest()

existing = {}
for split in ("valid", "test", "train"):
    for path in sorted((DATASET / split / "images").iterdir()):
        if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
            existing.setdefault(image_digest(path), (split, str(path)))
manifest = []
for path in negative_paths:
    digest = image_digest(path)
    if digest in existing:
        manifest.append({"source": str(path), "status": "skipped_duplicate",
                         "duplicate_of": existing[digest][1], "split": existing[digest][0]})
        continue
    stem = "hackathon_negative_" + digest
    destination = DATASET / "train/images" / (stem + path.suffix.lower())
    label = DATASET / "train/labels" / (stem + ".txt")
    assert not destination.exists() and not label.exists()
    shutil.copy2(path, destination)
    # Empty file = background image, not a new class.
    with label.open("x", encoding="utf-8") as f:
        pass
    existing[digest] = ("train", str(destination))
    manifest.append({"source": str(path), "status": "added", "split": "train",
                     "image": str(destination), "label": str(label)})
(DATASET / "hackathon_negative_manifest.json").write_text(json.dumps(manifest, indent=2))
print("Hackathon images:", len(negative_paths), "added:", sum(m["status"] == "added" for m in manifest),
      "duplicates skipped:", sum(m["status"] == "skipped_duplicate" for m in manifest))
print("Prepared data:", DATASET)


## 1. Audit the dataset before training
Checks paired files, polygon coordinates, class IDs, readable images, and identical images across splits. The dataset may contain near-duplicates that exact hashing cannot detect; inspect the previews too. Do not tune on the test split.


In [ ]:
source_config = yaml.safe_load((DATASET / "data.yaml").read_text())
names = source_config["names"]
if isinstance(names, dict):
    names = [names[i] for i in range(len(names))]
assert names == ["hard-toy", "soft-toy"], "This notebook requires the two toy classes."
print("Classes:", dict(enumerate(names)))
errors, duplicate_splits, summary = [], [], {}
seen = {}
for split in ("train", "valid", "test"):
    images = sorted(p for p in (DATASET / split / "images").iterdir()
                    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
    if not images:
        errors.append(f"No images in {split}")
    counts = Counter()
    empty = 0
    stems = set()
    for image_path in images:
        if image_path.stem in stems:
            errors.append(f"Repeated image stem: {image_path}")
        stems.add(image_path.stem)
        with Image.open(image_path) as im:
            im.verify()
        digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
        if digest in seen and seen[digest][0] != split:
            duplicate_splits.append([str(seen[digest][1]), str(image_path)])
        seen[digest] = (split, image_path)
        label_path = DATASET / split / "labels" / f"{image_path.stem}.txt"
        if not label_path.is_file():
            errors.append(f"Missing label: {label_path}")
            continue
        lines = label_path.read_text().splitlines()
        empty += not any(line.strip() for line in lines)
        for line in lines:
            if not line.strip():
                continue
            try:
                values = [float(x) for x in line.split()]
                assert len(values) >= 7 and len(values) % 2 == 1
                cls = int(values[0])
                assert values[0] == cls and 0 <= cls < len(names)
                assert all(math.isfinite(v) and 0 <= v <= 1 for v in values[1:])
                polygon = np.array(values[1:], dtype=np.float32).reshape(-1, 2)
                assert cv2.contourArea(polygon) > 0
                counts[names[cls]] += 1
            except (ValueError, AssertionError):
                errors.append(f"Invalid segmentation label: {label_path}")
    for label_path in (DATASET / split / "labels").glob("*.txt"):
        if label_path.stem not in stems:
            errors.append(f"Orphan label: {label_path}")
    summary[split] = {"images": len(images), "instances": dict(counts), "negative_images": empty}
print(json.dumps(summary, indent=2))
print("Cross-split exact duplicates:", len(duplicate_splits))
if duplicate_splits:
    print(duplicate_splits[:5])
assert not errors, "Dataset errors: " + str(errors[:10])
assert not duplicate_splits, "Resolve cross-split duplicate images before training; keep original data and create a cleaned copy."

## 2. Preview the training labels
Colored polygons should follow the toys. Inspect hard/soft class assignments and small or occluded toys before proceeding.


In [ ]:
rng = random.Random(SEED)
paths = sorted((DATASET / "train/images").glob("*"))
for path in rng.sample(paths, min(4, len(paths))):
    frame = cv2.imread(str(path))
    if frame is None:
        continue
    h, w = frame.shape[:2]
    for line in (DATASET / "train/labels" / f"{path.stem}.txt").read_text().splitlines():
        if not line.strip():
            continue
        values = list(map(float, line.split()))
        cls = int(values[0])
        polygon = (np.array(values[1:]).reshape(-1, 2) * [w, h]).astype(np.int32)
        cv2.polylines(frame, [polygon], True, (0, 200, 255), 2)
        x, y = polygon.min(axis=0)
        cv2.putText(frame, names[cls], (int(x), max(15, int(y))), cv2.FONT_HERSHEY_SIMPLEX, .5, (0, 0, 255), 1)
    preview = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    preview.thumbnail((650, 650))
    print(path.name)
    display(preview)


## 3. Write a local training configuration
The supplied YAML uses relative paths. Generate a new absolute-root configuration without modifying the source dataset. Keep validation and test splits in their correct roles.


In [ ]:
CONFIG_DIR = RUNS / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_YAML = CONFIG_DIR / f"{RUN_NAME}.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": DATASET.as_posix(), "train": "train/images", "val": "valid/images",
    "test": "test/images", "names": dict(enumerate(names))
}, sort_keys=False))
(CONFIG_DIR / f"{RUN_NAME}_audit.json").write_text(json.dumps(summary, indent=2))
print(DATA_YAML.read_text())


## 4. Load pretrained segmentation weights
This cell may download `yolo26n-seg.pt`. It is separate from the detection weights used by `test2.py`.


In [ ]:
PRETRAINED.parent.mkdir(parents=True, exist_ok=True)
model = YOLO(str(PRETRAINED))
assert model.task == "segment"
print("Loaded:", PRETRAINED)


## 5. Train
Run this cell to start GPU training. `workers=0` avoids Windows notebook multiprocessing issues. Early stopping monitors validation results. The best checkpoint is selected on validation, not test data. All classes are learned from the selected dataset; this custom model does not retain all 80 COCO categories.


In [ ]:
# A fresh name on every execution, even if this training cell is rerun.
TRAIN_NAME = f"{RUN_NAME}_{uuid.uuid4().hex}"
assert not (RUNS / TRAIN_NAME).exists()
training = model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH,
    device=DEVICE, workers=0, seed=SEED, patience=25, close_mosaic=10,
    project=str(RUNS), name=TRAIN_NAME, exist_ok=False,
    pretrained=True, cache=False, plots=True,
)
RUN_DIR = Path(model.trainer.save_dir)
BEST = RUN_DIR / "weights/best.pt"
assert BEST.is_file(), f"No best checkpoint at {BEST}"
print("Best trained weights:", BEST)

## 6. Evaluate validation performance
Review both box and mask metrics. A high average score may hide poor results on small toys. Inspect the saved confusion matrix and prediction plots. Tune using this split only.


In [ ]:
trained = YOLO(str(BEST))
validation = trained.val(data=str(DATA_YAML), split="val", imgsz=IMAGE_SIZE,
                         batch=BATCH, device=DEVICE, workers=0, plots=True, conf=0.001,
                         project=str(RUN_DIR), name="validation")
print("Box mAP50:", validation.box.map50, "Mask mAP50:", validation.seg.map50)
print("Box mAP50-95:", validation.box.map, "Mask mAP50-95:", validation.seg.map)

## Choose an inference threshold using validation only
Low-confidence validation (`conf=0.001`) retains predictions for AP curves; it is not the deployment threshold. Below, box precision/recall curves at IoU 0.5 select the highest-recall threshold meeting the configured mean precision and recall targets. If no threshold qualifies, use the best mean F1 and explicitly report the unmet target. Inspect each class too: a mean can hide one weak class.

The saved inference settings tie the threshold to this checkpoint and image size. A second validation run produces a confusion matrix with low-scoring predictions removed. Do not interpret its reported summary P/R as exact fixed-threshold metrics: Ultralytics also selects a point internally. Clothing-only validation images are needed to measure clothing false positives.

In [ ]:
x = np.asarray(validation.box.px)
p_curve = np.asarray(validation.box.p_curve)
r_curve = np.asarray(validation.box.r_curve)
mean_p, mean_r = p_curve.mean(axis=0), r_curve.mean(axis=0)
mean_f1 = np.asarray(validation.box.f1_curve).mean(axis=0)
eligible = np.flatnonzero((mean_p >= TARGET_PRECISION) & (mean_r >= MIN_RECALL))
if len(eligible):
    chosen = int(eligible[np.argmax(mean_r[eligible])])
    selection = "precision and recall targets met on validation"
else:
    chosen = int(np.argmax(mean_f1))
    selection = "targets NOT met; fallback to best mean F1"
CONFIDENCE = float(x[chosen])
print(selection)
print(f"Selected confidence={CONFIDENCE:.3f}, mean P={mean_p[chosen]:.3f}, mean R={mean_r[chosen]:.3f}")
print("Threshold    Mean precision    Mean recall")
for threshold in sorted(set([0.25, 0.45, 0.65, CONFIDENCE])):
    print(f"{threshold:.3f}        {np.interp(threshold, x, mean_p):.3f}             {np.interp(threshold, x, mean_r):.3f}")
for row, cls in enumerate(validation.box.ap_class_index):
    print(names[int(cls)], "P:", round(float(p_curve[row, chosen]), 3),
          "R:", round(float(r_curve[row, chosen]), 3))
settings = {"checkpoint": str(BEST.resolve()), "confidence": CONFIDENCE,
            "imgsz": IMAGE_SIZE, "data": str(DATA_YAML.resolve()),
            "selection": selection, "target_precision": TARGET_PRECISION,
            "minimum_recall": MIN_RECALL}
(RUN_DIR / "inference_settings.json").write_text(json.dumps(settings, indent=2))
operating_validation = trained.val(
    data=str(DATA_YAML), split="val", conf=CONFIDENCE, imgsz=IMAGE_SIZE,
    batch=BATCH, device=DEVICE, workers=0, plots=True,
    project=str(RUN_DIR), name="validation_selected_confidence")

# On verified toy-free validation images, every prediction is a false positive.
negative_paths = [p for p in sorted((DATASET / "valid/images").iterdir())
                  if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
                  and not (DATASET / "valid/labels" / (p.stem + ".txt")).read_text().strip()]
negative_report = []
for path in negative_paths:
    prediction = trained.predict(str(path), conf=CONFIDENCE, imgsz=IMAGE_SIZE,
                                 device=DEVICE, verbose=False)[0]
    negative_report.append({"image": str(path), "false_positives": len(prediction.boxes)})
if negative_report:
    print("Toy-free validation images:", len(negative_report),
          "false positives:", sum(r["false_positives"] for r in negative_report),
          "images with false positives:", sum(r["false_positives"] > 0 for r in negative_report))
else:
    print("No toy-free validation images: clothing rejection is not measured separately.")
(RUN_DIR / "negative_validation.json").write_text(json.dumps(negative_report, indent=2))

## 7. Final held-out test (after freezing your choices)
Only run after selecting your settings. This small test split does not establish general real-world accuracy. Record failures and evaluate additional footage from your intended camera.


In [ ]:
test_metrics = trained.val(data=str(DATA_YAML), split="test", imgsz=IMAGE_SIZE,
                           batch=BATCH, device=DEVICE, workers=0, plots=True, conf=0.001,
                           project=str(RUN_DIR), name="held_out_test")
report = {"checkpoint": str(BEST), "classes": names, "inference_settings": settings,
          "validation": validation.results_dict, "test": test_metrics.results_dict}
(RUN_DIR / "evaluation.json").write_text(json.dumps(report, indent=2, default=float))
print(json.dumps(report, indent=2, default=float))

## 8. Test one image with trained weights
Change the image path to a toy scene. CribHD-C can provide additional qualitative examples, but it is not a scored test without ground-truth labels.


In [ ]:
IMAGE_PATH = (REPO_ROOT / "CribHD/CribHD/CribHD-C/cribhdc08.jpg")
# CONFIDENCE comes from validation selection above.
with Image.open(IMAGE_PATH) as im:
    image = ImageOps.exif_transpose(im).convert("RGB")
result = trained.predict(image, conf=CONFIDENCE, imgsz=IMAGE_SIZE,
                         device=DEVICE, retina_masks=True, verbose=False)[0]
annotated = Image.fromarray(result.plot()[..., ::-1])
preview = annotated.copy()
preview.thumbnail((1000, 1000))
display(preview)
annotated.save(RUN_DIR / f"{IMAGE_PATH.stem}_prediction.png")
for box in result.boxes.cpu():
    print(result.names[int(box.cls.item())], round(float(box.conf.item()), 3))

## 9. Inspect toy mask areas (a diagnostic, not a danger classifier)
The source notebook uses fixed cutoffs of 8000 or 3000 mask pixels, which change meaning with resolution and camera distance. Show area fractions instead, without inventing a validated hazard cutoff. These remain image-space measurements, not physical size.


In [ ]:
if result.masks is None:
    print("No masks detected.")
else:
    masks = result.masks.data.cpu().numpy() > 0.5
    for i, mask in enumerate(masks):
        ys, xs = np.nonzero(mask)
        if not len(xs):
            continue
        centroid = (float(xs.mean()), float(ys.mean()))
        box = result.boxes[i]
        label = result.names[int(box.cls.item())]
        print(f"{label}: mask pixels={len(xs)}, image fraction={mask.mean():.5f}, centroid={centroid}")


In [ ]:
# Standalone test: run this cell after a kernel restart without retraining.
from pathlib import Path
# Resolve paths from the cloned repository (run the notebook from its root).
from repo_paths import find_repo_root
REPO_ROOT = find_repo_root()
from PIL import Image, ImageOps
from IPython.display import display
from ultralytics import YOLO
import torch
import json
import uuid

# After a restart, replace None with this experiment's printed best.pt path.
CHECKPOINT = globals().get("BEST", None)
assert CHECKPOINT is not None, "Set CHECKPOINT to the new experiment's best.pt first."
CHECKPOINT = Path(CHECKPOINT)
OUTPUT_ROOT = (REPO_ROOT / "training_runs_hackathon_negatives")
IMAGE_PATH = (REPO_ROOT / "CribHD/CribHD/CribHD-C/cribhdc08.jpg")
# Set CHECKPOINT above to the newly trained best.pt before standalone testing.
settings_path = CHECKPOINT.parent.parent / "inference_settings.json"
if settings_path.is_file():
    saved_settings = json.loads(settings_path.read_text())
    assert Path(saved_settings["checkpoint"]).resolve() == CHECKPOINT.resolve()
    CONFIDENCE = float(saved_settings["confidence"])
    IMAGE_SIZE = int(saved_settings["imgsz"])
else:
    CONFIDENCE, IMAGE_SIZE = 0.45, 640
    print("No saved calibration: using provisional confidence 0.45 at size 640.")
DEVICE = 0 if torch.cuda.is_available() else "cpu"
RUN_DIR = CHECKPOINT.parent.parent
trained = YOLO(str(CHECKPOINT))
print("Checkpoint:", CHECKPOINT)
print("Classes:", trained.names)
with Image.open(IMAGE_PATH) as im:
    image = ImageOps.exif_transpose(im).convert("RGB")
result = trained.predict(image, conf=CONFIDENCE, imgsz=IMAGE_SIZE,
                         device=DEVICE, retina_masks=True, verbose=False)[0]
annotated = Image.fromarray(result.plot()[..., ::-1])
preview = annotated.copy()
preview.thumbnail((1000, 1000))
display(preview)
prediction_dir = OUTPUT_ROOT / "standalone_predictions" / uuid.uuid4().hex
prediction_dir.mkdir(parents=True, exist_ok=False)
output = prediction_dir / f"{IMAGE_PATH.stem}_{IMAGE_SIZE}_{CONFIDENCE}_prediction.png"
annotated.save(output)
boxes = result.boxes.cpu()
print(f"Detected {len(boxes)} toy(s) at confidence >= {CONFIDENCE}, input size {IMAGE_SIZE}.")
if len(boxes) == 0:
    print("No predictions passed the threshold. This does NOT mean the image contains no toys.")
for box in boxes:
    print(result.names[int(box.cls.item())], round(float(box.conf.item()), 3),
          "box:", [round(v) for v in box.xyxy[0].tolist()])
print("Saved annotated image:", output)

## Adapting the original toy-filter pipeline later
- Replace `DefaultPredictor` with the trained YOLO26-seg checkpoint; use `result.boxes` and `result.masks` instead of Detectron2 `Instances`.
- MediaPipe face landmarks can still locate the mouth; Detectron2 is not needed for that step. No reliable face means proximity is unknown.
- Compute the mouth centroid from landmark coordinates and the toy centroid from mask pixels. The original final cell incorrectly calls the polygon-area function for centroids and passes a raster mask as polygon vertices.
- Keep image, mask and landmark coordinates aligned. Convert BGR to RGB before MediaPipe (one earlier source cell omits that conversion).
- Calibrate image-distance rules separately. A small nearby toy detection is not proof of choking risk, and no detection is not proof of safety.
- MiDaS is loaded in the original notebook but its predictions are not used in the shown filter; it is omitted here.

This notebook covers training and evaluation, not the full mouth-distance application. Keep `best.pt`, the run configuration, metrics, and dataset version. A segmentation checkpoint needs a mask-capable inference path; `test2.py` supports detection and segmentation checkpoints. Pass the new checkpoint with `--model`, the saved confidence with `--threshold`, and the saved image size with `--imgsz`.